# Stage 7 - LMS-Net (Zhang et al. 2025, Remote Sensing) - Colab Aug45 v2

Paper-faithful implementation target: **LMS-Net** (Hongyang Zhang, Shuo Liu, Huamei Liu, *Remote Sensing* 2025, 17, 227; DOI 10.3390/rs17020227).

## Deviations (must be reported)
- Input adapted to 14-band Sentinel-2 chips (paper dataset is satellite RGB imagery context).
- Binary mining mask output (1 channel sigmoid) for project masks.
- Channel schedule `[32,64,128,256,512]` made explicit (paper figure text not fully explicit in extraction).
- Stage 7 project protocol: 2023 GroupKFold-by-transect CV; 2024 held out report-only; flip/90° train-fold augmentation only; no leakage.

## Protocol
- 2024 excluded from model selection/early stopping.
- Augmentation only on training fold samples.
- Sequential single run.

In [ ]:
import json, os, random, platform
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
from scipy.ndimage import rotate as ndrotate, shift as ndshift
import pandas as pd
from PIL import Image
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
INPUT_DIR = Path(os.environ.get('YAMUNA_STAGE7_INPUT', '/kaggle/input/yamuna-stage7-input'))
CHIPS_DIR = Path(os.environ.get('YAMUNA_STAGE7_CHIPS', str(INPUT_DIR / 'chips_fixed')))
MASKS_DIR = Path(os.environ.get('YAMUNA_STAGE7_MASKS', str(INPUT_DIR / 'annotation' / 'masks')))
CHIPS_JSON = Path(os.environ.get('YAMUNA_STAGE7_CHIPS_JSON', str(INPUT_DIR / 'annotation' / 'chips.json')))
OUTPUT_DIR = Path(os.environ.get('YAMUNA_STAGE7_OUTPUT', '/content/stage7_lmsnet_aug45'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ARCH = 'lmsnet'
CHANNELS = [32, 64, 128, 256, 512]  # explicit deviation: paper figure channel schedule not fully explicit
EPOCHS = int(os.environ.get('STAGE7_EPOCHS', '40'))
BATCH_SIZE = int(os.environ.get('STAGE7_BATCH', '2'))
LR = 1e-3
SEED = 42
CV_SPLITS = 3
THRESH = 0.5

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)
print('paths', CHIPS_DIR.exists(), MASKS_DIR.exists(), CHIPS_JSON.exists())

## Paper-faithful modules

In [ ]:
class ChannelAttention(nn.Module):
    """CAM per paper Sec 3.4: max/avg pool -> shared MLP(C/8,C) -> add -> sigmoid -> scale."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.mlp = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False),
        )
    def forward(self, x):
        b,c,_,_ = x.shape
        g1 = F.adaptive_max_pool2d(x,1).view(b,c)
        g2 = F.adaptive_avg_pool2d(x,1).view(b,c)
        w = torch.sigmoid(self.mlp(g1) + self.mlp(g2)).view(b,c,1,1)
        return x * w

class LMSBlock(nn.Module):
    """LMS block per paper Sec 3.3/Fig 3.
    4 parallel 3x3 dilated convs (d=1,2,3,5), each out_ch/8 -> concat F5 (out_ch/2).
    Depthwise 3x3 on F5 -> concat [F5, DW(F5)] = out_ch.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        assert out_ch % 8 == 0, 'out_ch must be divisible by 8'
        branch_ch = out_ch // 8
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_ch, branch_ch, 3, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(branch_ch), nn.ReLU(inplace=True),
            ) for d in (1,2,3,5)
        ])
        f5_ch = branch_ch * 4  # out_ch/2
        self.dw = nn.Sequential(
            nn.Conv2d(f5_ch, f5_ch, 3, padding=1, groups=f5_ch, bias=False),
            nn.BatchNorm2d(f5_ch), nn.ReLU(inplace=True),
        )
        self.project = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        f5 = torch.cat([b(x) for b in self.branches], dim=1)
        f6 = self.dw(f5)
        return self.project(torch.cat([f5, f6], dim=1))

class LMSNet(nn.Module):
    """Paper-faithful 5-level encoder-decoder with LMS blocks + CAM."""
    def __init__(self, in_ch=14, channels=None, num_classes=1):
        super().__init__()
        ch = channels or CHANNELS
        assert len(ch) == 5
        self.pool = nn.MaxPool2d(2)
        self.enc1 = LMSBlock(in_ch, ch[0])
        self.enc2 = LMSBlock(ch[0], ch[1])
        self.enc3 = LMSBlock(ch[1], ch[2])
        self.enc4 = LMSBlock(ch[2], ch[3])
        self.enc5 = LMSBlock(ch[3], ch[4])
        self.cam1, self.cam2, self.cam3, self.cam4, self.cam5 = [ChannelAttention(c) for c in ch]
        # Decoder: Up(C5)+C4 -> D4 etc.
        self.dec4 = LMSBlock(ch[4] + ch[3], ch[3])
        self.dec3 = LMSBlock(ch[3] + ch[2], ch[2])
        self.dec2 = LMSBlock(ch[2] + ch[1], ch[1])
        self.dec1 = LMSBlock(ch[1] + ch[0], ch[0])
        self.head = nn.Sequential(
            nn.Conv2d(ch[0], ch[0], 3, padding=1, bias=False),
            nn.BatchNorm2d(ch[0]), nn.ReLU(inplace=True),
            nn.Conv2d(ch[0], num_classes, 1)
        )
    def up(self, x, size):
        return F.interpolate(x, size=size, mode='bilinear', align_corners=False)
    def forward(self, x):
        b1 = self.enc1(x)
        b2 = self.enc2(self.pool(b1))
        b3 = self.enc3(self.pool(b2))
        b4 = self.enc4(self.pool(b3))
        b5 = self.enc5(self.pool(b4))
        c1,c2,c3,c4,c5 = self.cam1(b1), self.cam2(b2), self.cam3(b3), self.cam4(b4), self.cam5(b5)
        d4 = self.dec4(torch.cat([self.up(c5, c4.shape[2:]), c4], dim=1))
        d3 = self.dec3(torch.cat([self.up(d4, c3.shape[2:]), c3], dim=1))
        d2 = self.dec2(torch.cat([self.up(d3, c2.shape[2:]), c2], dim=1))
        d1 = self.dec1(torch.cat([self.up(d2, c1.shape[2:]), c1], dim=1))
        return self.head(d1)

def bce_dice(logits, target):
    bce = F.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    inter = (prob*target).sum(dim=(2,3))
    dice = (2*inter+1)/(prob.sum(dim=(2,3))+target.sum(dim=(2,3))+1)
    return bce + (1-dice).mean()

## Data + protocol helpers

In [ ]:
entries = json.loads(CHIPS_JSON.read_text(encoding='utf-8'))
print('mask entries', len(entries))

def load_chip(e):
    p = CHIPS_DIR / e['quarter'] / f"{e['chip_id']}.tif"
    with rasterio.open(p) as ds: arr = ds.read().astype(np.float32)
    arr[:10] /= 10000.0; arr[10:] = arr[10:] / 5000.0 - 1.0
    return arr

def load_mask(e):
    p = MASKS_DIR / f"{e['chip_id']}_mask.png"
    with Image.open(p) as im: m = np.asarray(im.convert('L'), dtype=np.float32)
    if m.shape != (256,256): m = np.asarray(Image.fromarray(m.astype(np.uint8)).resize((256,256)), dtype=np.float32)
    return (m > 127).astype(np.float32)

from scipy.ndimage import rotate as ndrotate, shift as ndshift

def augment(x, y):
    """Train-fold-only augmentation: real + rotations 45/90/135/180 + pixel translations.
    Gate-compliant: applied only to training samples after split; never val/held-out/test.
    45/135 use reflect fill + interpolation; mask re-binarized. Artifacts documented."""
    out = [(x.copy(), y.copy())]  # real sample
    for angle in (45, 90, 135, 180):
        xr = ndrotate(x, angle, axes=(1, 2), reshape=False, order=1, mode='reflect').astype(np.float32)
        yr = ndrotate(y, angle, axes=(0, 1), reshape=False, order=0, mode='reflect')
        yr = (yr > 0.5).astype(np.float32)
        out.append((xr, yr))
    for dx, dy in ((-16, 0), (16, 0), (0, -16), (0, 16)):
        xr = ndshift(x, shift=(0, dy, dx), order=1, mode='reflect').astype(np.float32)
        yr = ndshift(y, shift=(dy, dx), order=0, mode='reflect')
        yr = (yr > 0.5).astype(np.float32)
        out.append((xr, yr))
    return out  # 1 real + 8 augmented = 9

def compute_metrics(y_true, y_prob, threshold=THRESH):
    pred = y_prob >= threshold; truth = y_true.astype(bool)
    tp=int((pred&truth).sum()); fp=int((pred&~truth).sum()); fn=int((~pred&truth).sum()); tn=int((~pred&~truth).sum())
    prec = tp/(tp+fp) if tp+fp else 0.0
    rec = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*prec*rec/(prec+rec) if prec+rec else 0.0
    iou = tp/(tp+fp+fn) if tp+fp+fn else 0.0
    dice = 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
    return dict(tp=tp,fp=fp,fn=fn,tn=tn,precision=prec,recall=rec,f1=f1,iou=iou,dice=dice,positive_fraction=float(truth.mean()))

cv_entries = [e for e in entries if not str(e['quarter']).startswith('2024')]
heldout_2024 = [e for e in entries if str(e['quarter']).startswith('2024')]
cache={e['chip_id']:(load_chip(e), load_mask(e)) for e in entries}
print('cv_entries', len(cv_entries), 'heldout_2024', len(heldout_2024))

## CV on 2023 mask chips

In [ ]:
# Gate-compliant: split on REAL 2023 chips first; augment training fold only; val real-only.
Xr, Yr, Gr, CHr = [], [], [], []
for e in cv_entries:
    x,y = cache[e['chip_id']]
    Xr.append(x); Yr.append(y); Gr.append(e['transect_id']); CHr.append(e['chip_id'])
Xr=np.stack(Xr); Yr=np.stack(Yr)[:,None]; Gr=np.asarray(Gr); CHr=np.asarray(CHr)
print('cv real dataset', Xr.shape, 'groups', len(set(Gr)))

fold_reports=[]
for fold,(tr_idx,va_idx) in enumerate(GroupKFold(n_splits=CV_SPLITS).split(Xr,Yr,Gr),1):
    Xtr,Ytr=[],[]
    for i in tr_idx:
        for xa,ya in augment(Xr[i], Yr[i,0]):
            Xtr.append(xa); Ytr.append(ya)
    Xtr=np.stack(Xtr); Ytr=np.stack(Ytr)[:,None]
    Xva=Xr[va_idx]; Yva=Yr[va_idx]
    mean=Xtr.mean(axis=(0,2,3),keepdims=True); std=Xtr.std(axis=(0,2,3),keepdims=True)+1e-6
    xtr=torch.from_numpy(((Xtr-mean)/std).astype(np.float32)); ytr=torch.from_numpy(Ytr.astype(np.float32))
    xva=torch.from_numpy(((Xva-mean)/std).astype(np.float32))
    loader=DataLoader(TensorDataset(xtr,ytr), batch_size=BATCH_SIZE, shuffle=True)
    model=LMSNet(in_ch=14, channels=CHANNELS).to(device)
    opt=torch.optim.Adam(model.parameters(), lr=LR)
    best_iou,best_state,best_epoch=-1,None,-1
    for epoch in range(1,EPOCHS+1):
        model.train(); loss_sum=0.0
        for xb,yb in loader:
            xb,yb=xb.to(device),yb.to(device)
            opt.zero_grad(); loss=bce_dice(model(xb),yb); loss.backward(); opt.step(); loss_sum+=loss.item()*len(xb)
        model.eval()
        with torch.no_grad(): p=torch.sigmoid(model(xva.to(device))).cpu().numpy()
        m=compute_metrics(Yva[:,0], p)
        if m['iou']>best_iou:
            best_iou,best_epoch=m['iou'],epoch
            best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
        if epoch%10==0 or epoch==1:
            print(f"fold {fold} epoch {epoch:2d} loss {loss_sum/len(xtr):.4f} val_iou {m['iou']:.4f} train_n {len(xtr)}")
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad(): prob=torch.sigmoid(model(xva.to(device))).cpu().numpy()
    m=compute_metrics(Yva[:,0], prob); m.update(fold=fold,best_epoch=best_epoch,val_chips=sorted(CHr[va_idx].tolist()), train_n_real=int(len(tr_idx)), train_n_aug=int(len(Xtr)))
    fold_reports.append(m)
    torch.save(dict(state=best_state,arch=ARCH,channels=CHANNELS,mean=mean,std=std,best_epoch=best_epoch), OUTPUT_DIR/f'fold_{fold}.pt')
    print('fold', fold, 'iou', m['iou'], 'dice', m['dice'])


## Final fit on 2023, held-out 2024 report-only

In [ ]:
Xall,Yall=[],[]
for e in cv_entries:
    x,y=cache[e['chip_id']]
    for xa,ya in augment(x,y): Xall.append(xa); Yall.append(ya)
Xall=np.stack(Xall); Yall=np.stack(Yall)[:,None]
mean=Xall.mean(axis=(0,2,3),keepdims=True); std=Xall.std(axis=(0,2,3),keepdims=True)+1e-6
xtr=torch.from_numpy(((Xall-mean)/std).astype(np.float32)); ytr=torch.from_numpy(Yall.astype(np.float32))
loader=DataLoader(TensorDataset(xtr,ytr), batch_size=BATCH_SIZE, shuffle=True)
model=LMSNet(in_ch=14, channels=CHANNELS).to(device); opt=torch.optim.Adam(model.parameters(), lr=LR)
for epoch in range(1,EPOCHS+1):
    model.train(); loss_sum=0.0
    for xb,yb in loader:
        xb,yb=xb.to(device),yb.to(device)
        opt.zero_grad(); loss=bce_dice(model(xb),yb); loss.backward(); opt.step(); loss_sum+=loss.item()*len(xb)
    if epoch%10==0 or epoch==1: print(f'final epoch {epoch:2d} loss {loss_sum/len(xtr):.4f}')
model.eval()
torch.save(dict(state={k:v.cpu().clone() for k,v in model.state_dict().items()},arch=ARCH,channels=CHANNELS,mean=mean,std=std), OUTPUT_DIR/'final_2023.pt')
heldout_rows=[]
for e in heldout_2024:
    x,y=cache[e['chip_id']]
    xin=torch.from_numpy(((x-mean.squeeze(0))/std.squeeze(0)).astype(np.float32)[None]).to(device)
    with torch.no_grad(): prob=torch.sigmoid(model(xin)).cpu().numpy()[0,0]
    m=compute_metrics(y, prob); m.update(chip_id=e['chip_id'], quarter=e['quarter'], transect_id=e['transect_id']); heldout_rows.append(m)
heldout_df=pd.DataFrame(heldout_rows)
heldout_summary={k:float(heldout_df[k].mean()) for k in ['iou','dice','precision','recall','f1']} if len(heldout_df) else {}
if len(heldout_df): heldout_df.to_csv(OUTPUT_DIR/'heldout_2024_predictions.csv', index=False)
overall={k:float(np.mean([f[k] for f in fold_reports])) for k in ['iou','dice','precision','recall','f1']}
metrics={
  'stage':7,'model':'LMSNet','arch':ARCH,'paper':'Zhang et al. 2025 Remote Sensing 17:227 doi:10.3390/rs17020227','channels':CHANNELS,
  'epochs':EPOCHS,'seed':SEED,'device':str(device),'timestamp_utc':datetime.now(timezone.utc).isoformat(),
  'n_mask_entries_total':len(entries),'n_cv_2023':len(cv_entries),'n_heldout_2024':len(heldout_2024),
  'cv_overall':overall,'cv_folds':fold_reports,'heldout_2024_summary_report_only':heldout_summary,
  'deviations':['Input adapted to 14-band S2 chips','Binary 1-channel mining mask output','Explicit channel schedule [32,64,128,256,512]','Stage 7 project protocol (2023 CV, 2024 heldout report-only, flip/90 train-only, no leakage)'],
  'protocol_notes':['2024 excluded from CV/model selection/early stopping; report only.','Augmentation (real + rot45/90/135/180 + pixel shifts ±16px x/y = 9x per training sample) on training fold samples only; 45/135 use reflect interpolation and mask re-binarization.','GroupKFold by transect on 2023 mask chips.','No leakage.','Sparse/tiny masks may yield near-zero IoU/Dice; report as-is.']
}
(OUTPUT_DIR/'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
pd.DataFrame(fold_reports).to_csv(OUTPUT_DIR/'cv_fold_metrics.csv', index=False)
print(json.dumps({'cv_overall':overall,'heldout_2024_summary':heldout_summary}, indent=2))